# 11B1 — Whole-atlas pseudobulk PCA and UMAP

Optional, read-only follow-on to 11B. No R fit, raw-cell I/O, segmentation or scVI rerun. Keep this notebook next to the **unchanged** 11A–11G notebooks. In particular, do not add a helper inside `ct26_atlas/`: the original atlas fingerprint includes the hashes of those files.

**Two distinct views:**
- Within each of the 20 populations: use saved DESeq2 VST from the completed population fit, with one point per eligible independent mouse. This is the preferred treatment/QC view. If all mice are eligible there are 12 points.
- Joint major-lineage and sibling-subtype views: use a common log2(CPM+1) normalization on the original cached pseudobulk sums, based on features measured in every included sample. The denominator is total counts across those common features (before variance/count filtering). These are expression-relationship views, not DESeq2 VST across independently fitted populations. The 9 major lineages form one disjoint view; each of the 4 subdivided lineages forms its own disjoint sibling view. Never stack lineage pools and their component cells in the same joint plot.

Top 500 variable genes by default; no significant-gene selection. PCA is centered without gene-wise variance scaling. UMAP uses up to ten nonzero PCs, `n_neighbors=5`, `min_dist=0.3`, fixed seed and random initialization. With only 12 mice per population, PCA and original expression distances are more interpretable than UMAP geometry. UMAP is exploratory and not a statistical test or a biological distance metric.

The notebook creates a new **companion evidence release**, with a filterable HTML browser, full coordinates, selected expression, PCA loadings, variance, distance matrices and eligibility/status. It uses the same catalogue format as 11G, but does not modify or automatically merge with an existing 11G release. The optional ZIP contains only processed pseudobulk outputs. `umap-learn` is optional for PCA; no dependencies are installed automatically.

In [ ]:
# 1 — Reopen the existing whole-atlas run without modifying its hashed helpers.
from pathlib import Path
import sys
from IPython.display import display
ADDON_DIR=Path.cwd()
# ADDON_DIR=Path('/path/to/extracted/CT26_11_Whole_Atlas_Evidence')
if not (ADDON_DIR/'ct26_atlas').is_dir():raise FileNotFoundError('Place this notebook alongside the original 11A–11G package or set ADDON_DIR.')
sys.path.insert(0,str(ADDON_DIR))
from ct26_atlas import core as ac
if Path(ac.__file__).resolve()!=(ADDON_DIR/'ct26_atlas'/'core.py').resolve():raise RuntimeError('Wrong helper imported; restart the kernel and set ADDON_DIR.')
ATLAS_ROOT=None
if ATLAS_ROOT is None:
    pointer=ADDON_DIR/'active_whole_atlas.json'
    if not pointer.is_file():raise FileNotFoundError('Complete 11A or set ATLAS_ROOT explicitly.')
    ATLAS_ROOT=Path(ac.load_json(pointer)['atlas_root'])
atlas=ac.reopen_atlas(ATLAS_ROOT)
MODEL_ROOT=ac.registered(atlas,'expression')
if MODEL_ROOT is None:raise RuntimeError('Complete 11B first. No fitted expression branch was registered.')
print('Existing atlas:',atlas['out'])
print('Selected 11B fit:',MODEL_ROOT)

## 2 — Embedded helper code
The following cell defines the functions; you do not need to edit it. It uses existing verified atlas helpers for source matching and the evidence catalogue, but does not edit their files.

In [ ]:
"""Supplement to 11A/11B. Stored VST within populations; joint raw-sum logCPM across populations."""
from pathlib import Path
import json, hashlib, html, importlib.metadata, warnings, zipfile, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from ct26_atlas import core as ac
from ct26_atlas import export as ex

TREATMENT_COLORS=dict(zip(ac.CONDITIONS,['#64748B','#507EB0','#DB8D38','#8557A3']))
SHAPES=dict(zip(ac.CONDITIONS,['o','s','^','D']))
DEFAULTS=dict(top_variable_genes=500,min_selected_genes=20,n_pcs_umap=10,n_neighbors=5,min_dist=.3,seed=17,dpi=300,run_umap=True,label_joint_points=False,complete_joint_mice_only=False,joint_scopes=True)


def check_meta(meta,atlas):
    required=['library_id','sample','independent_unit','condition','n_cells','library_counts']
    if not set(required).issubset(meta):raise ValueError('Unexpected fitted metadata: '+str(set(required)-set(meta)))
    if meta.library_id.duplicated().any() or meta['sample'].duplicated().any():raise ValueError('Duplicate fitted library/sample')
    ref=atlas['sources'].set_index('sample')
    if not set(meta['sample']).issubset(ref.index):raise ValueError('Unknown sample')
    aligned=ref.loc[meta['sample']]
    if not np.array_equal(meta['condition'].astype(str),aligned.condition.astype(str)) or not np.array_equal(meta.independent_unit.astype(str),aligned.unit.astype(str)):raise ValueError('Fitted study design differs from atlas')


def read_vst(atlas,model,pop):
    d=Path(model)/ac.slug(pop);receipt=ac.load_json(d/'complete.json')
    names=['vst_expression.tsv.gz','fitted_library_metadata.csv']
    for f in names:
        if receipt['hashes'].get(f)!=ac.sha(d/f):raise ValueError('Changed fitted input: '+str(d/f))
    meta=pd.read_csv(d/names[1],dtype={'library_id':str,'sample':str,'independent_unit':str,'condition':str},keep_default_na=False)
    check_meta(meta,atlas)
    x=pd.read_csv(d/names[0],sep='\t',index_col=0)
    x.index=x.index.astype(str);x.columns=x.columns.astype(str)
    if not x.index.is_unique or not x.columns.is_unique or set(x.columns)!=set(meta.library_id):raise ValueError('VST axes/metadata mismatch')
    if not np.isfinite(x.to_numpy(float)).all():raise ValueError('Nonfinite VST; no zero imputation')
    meta['population']=pop;meta['profile_id']=meta['sample']+'::'+pop
    x=x.loc[:,meta.library_id].T;x.index=meta.profile_id
    return x,meta,{f:ac.sha(d/f) for f in names}


def joint_input(atlas,groups,model_settings,complete_only):
    """One common transform for distinct populations, not a stack of independently fitted VSTs."""
    leafsets=[set(ac.GROUPS[g]) for g in groups]
    if any(a&b for i,a in enumerate(leafsets) for b in leafsets[i+1:]):raise ValueError('Joint view has overlapping populations')
    C,N=ac.group_arrays(atlas,groups);m=atlas['sources'];rows=[];arrays=[];avail=[]
    for i,rec in enumerate(m.to_dict('records')):
        for j,g in enumerate(groups):
            c=C[i,j];n=int(N[i,j]);total=int(c.sum());ok=n>=model_settings['min_cells'] and total>=model_settings['min_library_counts']
            rows.append(dict(profile_id=rec['sample']+'::'+g,sample=rec['sample'],independent_unit=rec['unit'],condition=rec['condition'],population=g,n_cells=n,library_counts=total,eligible_initial=ok))
            arrays.append(c);avail.append(atlas['P'][i])
    meta=pd.DataFrame(rows);eligible=meta.eligible_initial.to_numpy(bool)
    if complete_only:eligible &= meta['sample'].isin(meta.loc[meta.eligible_initial].groupby('sample').size().loc[lambda x:x==len(groups)].index).to_numpy()
    meta['included']=eligible
    if eligible.sum()<3 or meta.loc[eligible,'population'].nunique()!=len(groups):raise ValueError('Insufficient joint profiles or missing population after QC')
    a=np.asarray(arrays)[eligible];p=np.asarray(avail)[eligible].all(axis=0);den=a[:,p].sum(axis=1)
    if np.any(den<=0):raise ValueError('Zero common-feature library')
    keep=p & ((a>=model_settings['min_gene_count']).sum(axis=0)>=model_settings['min_gene_units'])
    x=np.log2(1+1e6*a[:,keep].astype(float)/den[:,None])
    use=meta.loc[eligible].copy();use['common_feature_library_counts']=den;use['library_id']=use.profile_id
    return pd.DataFrame(x,index=use.profile_id,columns=np.asarray(atlas['genes'])[keep]),use,meta


def ordinate(x,s,umap_module=None):
    if not x.index.is_unique or not x.columns.is_unique:raise ValueError('Duplicate axes')
    if len(x)<3 or not np.isfinite(x.to_numpy(float)).all():raise ValueError('Need ≥3 finite profiles')
    variance=x.var(axis=0,ddof=1);rank=pd.DataFrame({'gene':x.columns.astype(str),'variance':variance.to_numpy()}).sort_values(['variance','gene'],ascending=[False,True],kind='stable')
    selected=rank.loc[rank.variance>1e-12].head(s['top_variable_genes']).gene.tolist()
    if len(selected)<s['min_selected_genes']:raise ValueError('Too few variable genes')
    mat=x.loc[:,selected].astype(float);mean=mat.mean(axis=0);A=mat.to_numpy()-mean.to_numpy()
    U,D,V=np.linalg.svd(A,full_matrices=False);rank_n=int(np.sum(D>D[0]*1e-10))
    if rank_n<2:raise ValueError('Fewer than two nonzero principal components')
    U,D,V=U[:,:rank_n],D[:rank_n],V[:rank_n]
    # Fix arbitrary PCA signs for deterministic review; no treatment-dependent rotation.
    signs=np.sign(V[np.arange(rank_n),np.abs(V).argmax(axis=1)]);signs[signs==0]=1;U*=signs;V*=signs[:,None]
    pcs=U*D;names=[f'PC{i+1}' for i in range(rank_n)];coords=pd.DataFrame(pcs,index=mat.index,columns=names)
    ev=100*D**2/(D**2).sum();loadings=pd.DataFrame(V.T,index=selected,columns=names);loadings.insert(0,'centering_mean',mean.to_numpy());loadings.index.name='gene'
    rank['selected']=rank.gene.isin(selected);n_pc=min(rank_n,s['n_pcs_umap']);n_neighbors=min(s['n_neighbors'],len(mat)-1)
    u_status='disabled' if not s['run_umap'] else 'umap-learn unavailable' if umap_module is None else 'too_few_profiles' if len(mat)<5 else 'pending'
    coords['UMAP1']=np.nan;coords['UMAP2']=np.nan
    if u_status=='pending':
        try:
            u=umap_module.UMAP(n_components=2,n_neighbors=n_neighbors,min_dist=s['min_dist'],metric='euclidean',random_state=s['seed'],init='random',n_jobs=1).fit_transform(pcs[:,:n_pc])
            if not np.isfinite(u).all():raise ValueError('Nonfinite UMAP')
            coords[['UMAP1','UMAP2']]=u;u_status='complete'
        except Exception as e:u_status='failed: '+str(e)
    return dict(matrix=mat,coordinates=coords,loadings=loadings,feature_ranking=rank,variance=pd.DataFrame({'PC':names,'explained_variance_percent':ev}),distance=pd.DataFrame(cdist(mat,mat),index=mat.index,columns=mat.index),umap_status=u_status,n_pcs_umap=n_pc,n_neighbors=n_neighbors)


def draw(coords,ev,path,title,transform,s,joint):
    has_umap=coords[['UMAP1','UMAP2']].notna().all().all()
    axes_pairs=[('PC1','PC2')]+([('UMAP1','UMAP2')] if has_umap else [])
    ncols=2 if joint else len(axes_pairs);nrows=len(axes_pairs) if joint else 1
    fig,axs=plt.subplots(nrows,ncols,squeeze=False,figsize=(7.5*ncols,5.7*nrows))
    for i,(xx,yy) in enumerate(axes_pairs):
        views=['condition','population'] if joint else ['condition']
        for j,view in enumerate(views):
            ax=axs[i,j] if joint else axs[0,i]
            for c in ac.CONDITIONS:
                sub=coords[coords.condition==c]
                if sub.empty:continue
                if view=='condition':
                    ax.scatter(sub[xx],sub[yy],s=52,c=TREATMENT_COLORS[c],marker=SHAPES[c],edgecolors='black',linewidths=.45,label=ac.DISPLAY[c],alpha=.85)
                else:
                    ax.scatter(sub[xx],sub[yy],s=52,c=[ac.COLORS[g] for g in sub.population],marker=SHAPES[c],edgecolors='black',linewidths=.45,alpha=.85)
                if not joint or s['label_joint_points']:
                    for r in sub.itertuples():ax.annotate(r.sample,(getattr(r,xx),getattr(r,yy)),fontsize=8,xytext=(4,3),textcoords='offset points')
            if view=='condition':ax.legend(loc='best',frameon=False,fontsize=8)
            else:
                pops=list(dict.fromkeys(coords.population));h=[Line2D([0],[0],marker='o',ls='',markerfacecolor=ac.COLORS[g],markeredgecolor='black',label=g) for g in pops]
                h += [Line2D([0],[0],marker=SHAPES[c],ls='',color='0.3',label=ac.DISPLAY[c]) for c in ac.CONDITIONS]
                ax.legend(handles=h,loc='upper left',bbox_to_anchor=(1,1),fontsize=8,frameon=False)
            label=lambda z:z+(f' ({ev.loc[z]:.1f}%)' if z in ev.index else '')
            ax.set(xlabel=label(xx),ylabel=label(yy),title=('PCA' if xx=='PC1' else 'UMAP')+' | '+('treatment' if view=='condition' else 'population; shapes = treatment'));ax.spines[['top','right']].set_visible(False)
    fig.suptitle(title+' | '+transform+'\nOne point = one mouse × population pseudobulk',fontsize=13)
    fig.text(.02,.012,'Exploratory ordination; no significance test or batch correction. UMAP distances are not quantitative expression distances.',fontsize=9)
    fig.tight_layout(rect=[0,.04,1,.92]);ac.savefig(fig,path,s['dpi'])


def run(atlas,model_root,settings=None,output_parent=None):
    s={**DEFAULTS,**(settings or {})};model=Path(model_root).resolve();st=pd.read_csv(model/'fit_status.csv',keep_default_na=False)
    if st.population.duplicated().any() or set(st.population)-set(ac.POPS):raise ValueError('Invalid model status')
    spec0=ac.load_json(model/'settings.json');ms=spec0['settings']['model']
    if spec0['atlas_id']!=atlas['out'].name:raise ValueError('Model is from another atlas')
    if any(int(s[k])<2 for k in ['top_variable_genes','min_selected_genes','n_pcs_umap','n_neighbors']):raise ValueError('Invalid ordination settings')
    umap_module=None;umap_error=''
    if s['run_umap']:
        try:import umap as umap_module
        except Exception as e:umap_error=str(e);print('UMAP unavailable; PCA and expression distances will still be saved:',e)
    versions={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','matplotlib']}
    if umap_module is not None:versions['umap-learn']=importlib.metadata.version('umap-learn')
    spec=dict(version='11B1-v1',atlas_id=atlas['out'].name,model_id=model.name,model_fit_status_sha256=ac.sha(model/'fit_status.csv'),settings=s,software=versions,umap_import_error=umap_error)
    parent=Path(output_parent) if output_parent else atlas['out']/'pseudobulk_ordination'
    root=parent/('11B1_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+ac.digest(spec)[:10]);root.mkdir(parents=True,exist_ok=False)
    r=ex.Release(root,atlas,1024*2**20);statuses=[]
    # Same catalogue schema as 11G, with this supplement's own assets and no raw cell data.
    for f,desc in [('mouse_design.csv','Mouse-level design'),('hierarchy.csv','Annotation hierarchy and colors'),('annotation_coverage.csv','Reviewed-cell coverage'),('population_counts.csv','Population cell and library totals')]:r.data(atlas['out']/f,'data/study/'+f,'study',desc)
    r.generated_table(pd.DataFrame({'condition':ac.CONDITIONS,'display_name':[ac.DISPLAY[c] for c in ac.CONDITIONS],'order':range(4)}),'data/study/treatment_order.csv','study','Categorical condition order')
    def panel(name,pop,x,meta,transform,joint,source_hashes,audit=None):
        data=ordinate(x,s,umap_module);p=Path('data/ordination')/name;d=root/p;d.mkdir(parents=True,exist_ok=True)
        coords=meta.set_index('profile_id').join(data['coordinates'],validate='one_to_one').reset_index()
        if len(coords)!=len(x):raise ValueError('Ordination metadata join mismatch')
        files=[]
        for f,table,idx,desc in [('coordinates.csv',coords,False,'Exact plotted coordinates and mouse/population metadata'),('selected_expression.tsv.gz',data['matrix'],True,'Exact unscaled selected expression matrix used for PCA'),('gene_variance_and_selection.csv',data['feature_ranking'],False,'Unsupervised variance selection, not DE significance'),('PCA_loadings.tsv.gz',data['loadings'],True,'PCA loadings and centering means'),('PCA_variance.csv',data['variance'],False,'Explained variance of selected-gene matrix'),('expression_Euclidean_distances.tsv.gz',data['distance'],True,'Euclidean distances on selected expression features; not UMAP distances')]:files.append(r.generated_table(table,(p/f).as_posix(),'ordination',desc,index=idx))
        if audit is not None:files.append(r.generated_table(audit,(p/'library_eligibility.csv').as_posix(),'ordination','Included and excluded profile inventory'))
        settings_out=dict(transform=transform,joint=joint,n_profiles=len(coords),n_genes=data['matrix'].shape[1],umap_status=data['umap_status'],effective_n_neighbors=data['n_neighbors'],effective_n_pcs_umap=data['n_pcs_umap'],source_hashes=source_hashes,**s)
        rec=d/'ordination_settings.json';ac.save_json(settings_out,rec);rel=r.generated(rec);r.add_dataset(rel,'ordination','Parameters, source hashes and UMAP status');files.append(rel)
        f=root/'figures'/'ordination'/name/'PCA_UMAP.png';f.parent.mkdir(parents=True,exist_ok=True)
        draw(coords,data['variance'].set_index('PC').explained_variance_percent,f,pop if data['umap_status']=='complete' else pop+' (PCA only)',transform,s,joint)
        rel=r.generated(f);r.evidence.append(dict(evidence_id='F-'+ac.digest(rel)[:12],path=rel,title=pop+(' | pseudobulk PCA / UMAP' if data['umap_status']=='complete' else ' | pseudobulk PCA only'),chapter='Expression ordination',population=pop,lineage=pop if pop in ac.LINEAGES else ac.PARENT.get(pop,'Across atlas'),plot_kind='pseudobulk_ordination',meaning='One point per eligible mouse/population. PCA on top variable genes; UMAP on leading PCs if available. '+transform,caution='Exploratory only; no treatment significance. UMAP geometry and separate embeddings are not quantitative comparable distances; pooled and leaf profiles overlap across separate views.',status='available',data_paths=files,filters={'scope':name,'population':pop,'joint':joint},settings=settings_out,sha256=ac.sha(f)))
        return dict(scope=name,population=pop,status='complete',n_profiles=len(coords),n_genes=data['matrix'].shape[1],umap_status=data['umap_status'])
    for pop in ac.POPS:
        row=st.loc[st.population==pop]
        if row.empty or row.iloc[0].status!='complete':statuses.append(dict(scope='population_'+ac.slug(pop),population=pop,status='no_completed_fit',umap_status='not_run'));continue
        name='population_'+ac.slug(pop)
        try:
            x,meta,h=read_vst(atlas,model,pop);statuses.append(panel(name,pop,x,meta,'DESeq2 VST (within this population)',False,h))
        except Exception as e:statuses.append(dict(scope=name,population=pop,status='failed: '+str(e),umap_status='not_run'))
        print(statuses[-1],flush=True)
    if s['joint_scopes']:
        scopes=[('joint_major_lineages','Across atlas',ac.LINEAGES)]+[('joint_'+ac.slug(g),g,m) for g,m in ac.MULTI.items()]
        for name,pop,groups in scopes:
            try:
                x,meta,audit=joint_input(atlas,groups,ms,s['complete_joint_mice_only']);statuses.append(panel(name,pop,x,meta,'Joint log2(CPM + 1), common measured genes',True,{'leaf_pseudobulk.npz':ac.sha(atlas['out']/'leaf_pseudobulk.npz')},audit))
            except Exception as e:statuses.append(dict(scope=name,population=pop,status='not_available: '+str(e),umap_status='not_run'))
            print(statuses[-1],flush=True)
    status=pd.DataFrame(statuses);r.generated_table(status,'analysis_status.csv','ordination','Panel status, including unavailable or failed UMAPs')
    manifest=dict(release_id=root.name,**spec,n_figures=len(r.evidence),n_datasets=len(r.datasets),scope='Pseudobulk ordination companion only; no cell-level data')
    ac.save_json(manifest,root/'release_manifest.json');r.generated(root/'release_manifest.json');r.write_indices()
    doc='''# Pseudobulk ordination companion (11B1)\n\nOpen index.html and select a population. Each point represents one mouse/population pseudobulk, not a cell. Within-population plots use the completed DESeq2 VST (blind=FALSE) and all fitted eligible mice. Top variable genes selected by variance, never DEG significance; PCA is centered, not gene-z-scored. Joint plots use one common log2(CPM+1) transform from raw sums, with common measured features as library denominator. Do not concatenate independently fitted VSTs. Joint plots omit ineligible libraries and preserve their status. The major-lineage joint view is disjoint; sibling views are disjoint within each parent. They share mice across populations, not independent replicates. PCA variance applies only to selected features. UMAP on leading PCs is exploratory, sensitive to hyperparameters and seed; 12 profiles is a small sample. Quantitative distance table is Euclidean in the exact selected expression features. No clustering, significance tests, batch correction, or decontamination is done. Separately fitted axes are not comparable across populations. If UMAP is unavailable, disabled or fails, only PCA is shown and the actual status is recorded; no substitute layout is presented as UMAP. Check analysis_status.csv and input eligibility. All outputs link exact input matrices, coordinates, feature lists and settings. This is a supplement to, not an automatic modification of, the frozen 11G release.\n'''
    for name in ['START_HERE.md','AGENT_GUIDE.md','METHODS_AND_REFERENCES.md']:
        (root/name).write_text(doc+'\nReferences: https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html ; https://umap-learn.readthedocs.io/en/latest/parameters.html\n');r.generated(root/name)
    (root/'REVIEW_EXAMPLES.py').write_text("from pathlib import Path\nimport pandas as pd\nroot=Path.cwd()\nprint(pd.read_csv(root/'evidence_index.csv')[['population','plot_kind','path']])\n");r.generated(root/'REVIEW_EXAMPLES.py')
    ex.make_portal(root,r.evidence,r.datasets,[dict(stage='ordination',status='complete_with_audit',run_id=root.name,notes='See analysis_status.csv for panel and UMAP status')]);ex.validate_release(root,r.evidence,r.datasets)
    # Include only this notebook's actual source and lightweight results; no original model objects.
    print('Ordination companion:',root);return root,status


def freeze(root,notebook_path=None,make_zip=True):
    root=Path(root)
    if notebook_path is not None:
        p=Path(notebook_path)
        if p.is_file():
            nb=json.loads(p.read_text())
            for cell in nb['cells']:
                if cell['cell_type']=='code':cell['execution_count']=None;cell['outputs']=[]
            (root/'11B1_method_notebook.ipynb').write_text(json.dumps(nb,indent=1))
    files=[p for p in sorted(root.rglob('*')) if p.is_file() and p.name!='SHA256SUMS.txt'];(root/'SHA256SUMS.txt').write_text(''.join(ac.sha(p)+'  '+p.relative_to(root).as_posix()+'\n' for p in files))
    if not make_zip:return None
    out=root.with_suffix('.zip')
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for f in sorted(root.rglob('*')):
            if f.is_file():z.write(f,f.relative_to(root).as_posix())
    with zipfile.ZipFile(out) as z:
        if z.testzip():raise ValueError('Archive integrity failure')
    return out

In [ ]:
# 3 — Ordination settings and dependency check.
ORDINATION_SETTINGS=dict(
    top_variable_genes=500,
    min_selected_genes=20,
    n_pcs_umap=10,
    n_neighbors=5,
    min_dist=0.3,
    seed=17,
    dpi=300,
    run_umap=True,
    label_joint_points=False,
    complete_joint_mice_only=False,
    joint_scopes=True,
)
# False keeps all eligible libraries in joint views; true keeps only mice with all
# groups in that view. The eligibility audit records the excluded rows either way.
# The per-population plots always retain the actual eligible fitted mice from 11B.
if ORDINATION_SETTINGS['run_umap']:
    try:
        import umap
        print('UMAP version:',umap.__version__)
    except Exception as e:
        print('UMAP is unavailable in this kernel:',e)
        print('PCA can proceed. If needed, install umap-learn in your existing UV project, then restart the kernel.')

In [ ]:
# 4 — Render the population and joint views; inspect every panel's status.
ORDINATION_ROOT,ordination_status=run(atlas,MODEL_ROOT,ORDINATION_SETTINGS)
display(ordination_status)
print('Browser:',ORDINATION_ROOT/'index.html')
print('Evidence catalogue:',ORDINATION_ROOT/'evidence_index.csv')
print('Source plots and previous 11G releases remain unchanged.')

## 5 — Freeze the companion package
This is a separate lightweight release, not a replacement for 11G. It uses the same evidence/dataset schemas and can be filtered with the original `make_review_packet.py`. It contains no per-cell expression or coordinates. The optional notebook source snapshot strips saved outputs. If you only want the figures locally, set `MAKE_ZIP=False`.

In [ ]:
MAKE_ZIP=True
NOTEBOOK_SOURCE=ADDON_DIR/'11B1_CT26_Pseudobulk_PCA_UMAP.ipynb'
ORDINATION_ZIP=freeze(ORDINATION_ROOT,NOTEBOOK_SOURCE,make_zip=MAKE_ZIP)
print('Portable companion:',ORDINATION_ZIP)
# Optional narrow evidence packet from this companion:
# from make_review_packet import build_packet
# QUESTION_ZIP=build_packet(ORDINATION_ROOT,mode='question',population='All T cell',plot_kind='pseudobulk_ordination',max_figures=4,max_mib=120)

## Interpretation

Start with All tumor cell and the other major lineage panels. Compare PCA, UMAP, cell count/library metadata and selected-expression Euclidean distances before attributing separation to treatment. The existing VST used `blind=False` (a design-aware dispersion trend), not treatment removal or blind QC. These plots do not establish statistically significant differences. Missing/failed population fits are explicit; joint views use eligible raw pseudobulks even if a treatment DEG fit is untestable. The same mouse appears across populations and is not a new biological replicate. No cell-level correction or decontamination is performed.

Separate embeddings have different axes, and distances between panels are not comparable. An apparent outlier deserves review but is not automatically excluded. The 500-gene selection is exploratory and editable. For a more stringent sensitivity check, compare a different feature count, UMAP neighbor count/seed, or the saved expression distance matrix. References: [DESeq2 transformations](https://www.bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html) and [UMAP parameters](https://umap-learn.readthedocs.io/en/latest/parameters.html).

Validation: synthetic 12-mouse fixtures exercised VST metadata/hash matching, PCA reconstruction, common-feature normalization and missing-subtype eligibility, figure/linked-data export and ZIP integrity. `umap-learn` was unavailable in the authoring runtime, so the actual UMAP optimizer and actual CT26 data were not run. On-server results are recorded per panel.